# W1D4：数据进出与文本处理——从文件来，到文件去

**今天要解决的问题**：真实数据不在代码里——在CSV文件、JSON配置文件、TXT日志里。你得学会读进来、清洗干净、写回去。

**你要用到的武器**：`with open()` + `csv` + `json` + 字符串方法 + `re`正则

**需要的前置知识**：
- W1D1：list/dict/set/tuple —— 文件内容最终装进容器
- W1D2：if/for/while —— 读出来的数据用循环逐行处理
- W1D3：def函数 —— 把文件读写封装成可复用的函数

---

### 📋 今日知识地图

学完今天的内容后，你应该能逐项打勾：

**文件读写基础**
- [ ] `open(path, mode, encoding='utf-8')` + `with`语句（自动关闭）
- [ ] mode：`'r'`读 / `'w'`写（覆盖！）/ `'a'`追加 / `'r+'`读写
- [ ] `f.read()` / `f.readline()` / `f.readlines()` / `f.write()` / `f.writelines()`
- [ ] 理解为什么`with`比手动`f.close()`安全

**CSV处理**
- [ ] `csv.DictReader(f)` → 每行自动变dict（第一行=表头=key）
- [ ] `csv.DictWriter(f, fieldnames=[])` + `writeheader()` + `writerows()`
- [ ] ⚠️ CSV读进来的数字是字符串！必须 `int()` / `float()` 转换

**JSON处理**
- [ ] `json.load(f)` 读JSON文件 → dict/list
- [ ] `json.dump(obj, f, ensure_ascii=False)` 写JSON文件
- [ ] `json.loads(str)` / `json.dumps(obj)` 字符串与对象互转

**字符串方法全量**
- [ ] 去空格：`.strip()` / `.lstrip()` / `.rstrip()`
- [ ] 分割合并：`.split()` / `.rsplit()` / `.splitlines()` / `.join()`
- [ ] 查找替换：`.find()` / `.rfind()` / `.index()` / `.replace()` / `.count()`
- [ ] 判断类：`.startswith()` / `.endswith()` / `.isalpha()` / `.isdigit()` / `.isalnum()` / `.isspace()`
- [ ] 大小写：`.lower()` / `.upper()` / `.title()` / `.capitalize()` / `.swapcase()`
- [ ] f-string格式化：`:.2f` / `:>10` / `:.1%`

**正则表达式**
- [ ] `re.search()` / `re.findall()` / `re.sub()` / `re.split()` / `re.compile()`
- [ ] 元字符：`.` `^` `$` `*` `+` `?` `{n,m}` `[...]` `[^...]` `|` `()` `\d` `\w` `\s` `\b`
- [ ] 贪婪 vs 非贪婪：`.*`（贪）vs `.*?`（不贪）
- [ ] 分组与捕获：`()`捕获组 / `(?:...)`不捕获 / `\1`反向引用
- [ ] 前瞻后顾：`(?=...)`正前瞻 / `(?!...)`负前瞻 / `(?<=...)`正后顾
- [ ] 标志：`re.DOTALL` / `re.IGNORECASE` / `re.MULTILINE`
- [ ] ⚠️ `re.match`只从开头匹配 → 用`re.search`替代

---


## 🔴 认知冲突：数据写死在代码里——能撑到第几行？

前三天你一直在代码里手写数据：

```python
orders = [
    {"订单号": "ORD001", "金额": 299, "城市": "北京"},
    {"订单号": "ORD002", "金额": 1599, "城市": "上海"},
    ...  # 手写10笔还行
]
```

然后小王在钉钉上叫你：

> 小王：实习生，上个月的真实订单在 `orders_202401.csv` 里，大概5000行。还有产品配置是JSON格式的，用户行为日志是txt。你读一下，清洗干净，下午三点前把分析结果写到一个新CSV里，运营要导入Excel看。

你打开 `orders_202401.csv`，看到：

```
订单号,商品,金额,城市
ORD001,蓝牙耳机,299元, 北京
ORD002,机械键盘,待定,上海
ORD003,,89,北京
ORD004,充电宝,499,  深圳
ORD005,显示器,2599,上海
...5000行
```

你立刻发现了三个问题：

1. **5000行不可能手写到代码里** —— 必须用程序读文件
2. **数据很脏** —— 金额里混了"元"字、有的写"待定"、商品为空、城市前后有空格
3. **结果要写回文件** —— 运营要CSV格式，不是print在屏幕上

**前三天你学会了"处理内存里的数据"，今天你要学会"让数据从文件流进来、再流出去"。**


## 📦 回到速卖通

昨天你用函数封装了订单分析逻辑——`calc_total()`、`judge_order()`、推导式筛选。小王看了说"不错，有模块化的意识了"。

> 小王：行了，光会处理内存里的数据不够。我把真实数据文件发你——`orders.csv`（订单）、`config.json`（产品配置）、`event_log.txt`（用户行为日志）。你先读进来，然后我发现数据有点脏——金额那栏有人写了"元"字、有人写了"待定"、城市前后有空格。你洗干净了写回一个新CSV。对了，从日志里帮我提取所有手机号和邮箱。

今天你的任务很明确：**读文件 → 清洗文本 → 写文件**。这三个动作加在一起，就是你以后每天都要做的事情。


---

## 概念一：文件读写基础——`with open()` 是你的自动门

### 🤔 你现有的知识够用吗？

到目前为止，你所有的数据都在代码里——变量赋值、list字面量、dict字面量。程序一结束，数据全没了。

现在小王给你一个文件，里面有5000行订单。你怎么把它"弄进"Python？反过来——你分析完的结果怎么"保存到"文件里，让运营同事用Excel打开？

你需要的是一种"管道"——能把文件内容流进Python变量，也能把Python变量流出到文件。`open()`就是这根管道。


### 文件读写基础

**① 说白了就是**：打开文件就像进图书馆——你推开门（`open()`），找到书（读），或放一本书上架（写），然后**必须关门**（`close()`）。`with`语句就像自动门——进去时自动开，出来时自动关，永远不会忘。手动关门（`f.close()`）的问题是一旦中间代码报错，`close()`被跳过→文件锁着→别人打不开。

**② 在沙盘里怎么用**：`with open('orders.csv', 'r', encoding='utf-8') as f:` → 打开订单文件 → `content = f.read()` 全读进来 → `with`块结束自动关闭。写入：`with open('report.csv', 'w', encoding='utf-8') as f:` → `f.write(报告内容)` → 自动关闭。

**③ 跟你D3学的函数的关系**：D3教你封装逻辑——今天你可以把"读文件"封装成 `def read_orders(path):`，以后任何地方要读订单只需一行 `orders = read_orders('orders.csv')`。函数 + 文件IO = 可复用的数据管道。

**④ 为什么这样设计**：
- **`with`是上下文管理器** —— Python解决"资源管理"的优雅方案。`with`块无论正常结束还是抛异常，都会执行清理（关闭文件）。这比`try/finally`简洁得多。
- **三个关键mode的区别**：`'r'`读（文件必须存在，否则`FileNotFoundError`）、`'w'`写（⚠️**一打开就清空原文件内容**——这是今天第一个大坑）、`'a'`追加（写到文件末尾，不清空）。`'r+'`读写——指针在开头，写会覆盖已有内容。
- **`encoding='utf-8'`不能省**：Windows默认用GBK编码，如果你的文件是UTF-8（大多数情况），不写encoding → 中文乱码。养成肌肉记忆。
- **写CSV时加`newline=''`**：Windows上`\n`会被自动转成`\r\n`，在CSV里产生多余空行。加`newline=''`阻止这个行为。


In [1]:
# === 文件读写——它长这样 ===

# 写文件（⚠️ 'w'模式一打开就清空！）
with open('sales.txt', 'w', encoding='utf-8') as f:
    f.write('12\n8\n15\n')   # 三个数字，每行一个

# 读文件（with自动关闭）
with open('sales.txt', 'r', encoding='utf-8') as f:
    content = f.read()        # 全读成一个字符串
print(repr(content))          # '12\n8\n15\n'

# 逐行读（大文件推荐——不会一次性占满内存）
with open('sales.txt', 'r', encoding='utf-8') as f:
    for line in f:            # 文件对象本身可迭代！
        print(f'行内容: {line.strip()}')

# 追加模式——不清空，加到末尾
with open('sales.txt', 'a', encoding='utf-8') as f:
    f.write('22\n')           # 文件现在有4行

# 验证：读回来看看
with open('sales.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()      # 每行作为一个元素→list of str
    print(f'共{len(lines)}行: {lines}')


'12\n8\n15\n'
行内容: 12
行内容: 8
行内容: 15
共4行: ['12\n', '8\n', '15\n', '22\n']


In [9]:
# ========== 【文件读写小练】==========
# 任务：
# 1. 用'w'模式创建文件 numbers.txt，写入三行：100, 200, 300
# 2. 用'r'模式读回来，split()分割→转int→求和→打印
# 预期输出：总和：600

# 👇 你的代码：
with open("numbers.txt","w",encoding= "utf-8") as f:
    f.write("100\n200\n300\n")

with open("numbers.txt","r",encoding= "utf-8") as f:
    nums = [int(x) for x in f.read().split()] 
    print(f"总和：{sum(nums)}")


总和：600


---

## 概念二：CSV处理——数据分析的"通用语"

### 🤔 读进来是一坨字符串——怎么变回表格？

你用`f.read()`读进来的是一个大字符串：`"订单号,商品,金额,城市\nORD001,蓝牙耳机,299元,北京\n..."`

要用它——你得手动`split(',')`拆每一行、手动对第一行（表头）和后面各行做映射……写20行代码才能把CSV变成list of dict。这还没处理引号内逗号、换行符在字段里的边界情况。

**Python标准库有没有"CSV→list of dict"一行搞定？** 有——`csv.DictReader`就是专门做这件事的。


### CSV处理

**① 说白了就是**：CSV就是"用逗号分隔的表格"——Excel能打开、Python能读、数据库能导入。`csv.DictReader`自动把第一行当表头，后面每一行变成一个dict——"订单号"那列的值=dict的`"订单号"`key对应的value。一条CSV行 → 一个dict，整个CSV就是一个list of dict——这不就是你D1学的核心数据结构吗？

**② 在沙盘里怎么用**：`for row in csv.DictReader(f):` → 每一行自动转dict → `row['金额']`直接取金额列 → 然后你D2学的for+if、D3学的函数、D1学的容器全部可以上场了。CSV是数据进入你的Python世界的"海关通道"。

**③ 跟你D1学的list[dict]的关系**：D1你手写了list of dict——`[{"订单号": "ORD001", ...}, ...]`。`csv.DictReader`做的事完全一样——只不过数据不是手写的，是从文件流进来的。一个5000行的CSV → 就是一个5000个元素的大list，每个元素是一个dict。你D1的处理list[dict]的代码，现在可以原样用在实际数据上了！

**④ 为什么这样设计**：
- **DictReader的设计理念**：让CSV每一行像数据库的一行记录——用列名（表头）做key，行值做value。程序员不需要记"金额是第3列"，只需写`row['金额']`。表头改名？代码不用改。列顺序变了？不受影响。
- **⚠️ 最重要的陷阱**：CSV没有类型信息——读进来的**所有值都是字符串**！`row['金额']`是`"299"`（str），不是`299`（int）。必须`int(row['金额'])`手动转换。这个坑无数新手踩过——金额相加变成了字符串拼接（`'299'+'89'='29989'`）。
- **`DictWriter` + `writeheader()`**：写CSV时，`writeheader()`用你指定的`fieldnames`自动写表头行，`writerows()`把list of dict批量写入。别忘了`newline=''`——Windows上不写这个会在行间插入空行。


In [10]:
# === CSV处理——它长这样 ===
import csv

# 示例数据（模拟小王给的orders.csv内容）
orders = [
    {"订单号": "ORD001", "商品": "蓝牙耳机", "金额": 299, "城市": "北京"},
    {"订单号": "ORD002", "商品": "机械键盘", "金额": 1599, "城市": "上海"},
    {"订单号": "ORD003", "商品": "手机壳",   "金额": 89,  "城市": "北京"},
]

# 写CSV（⚠️ newline=''防Windows空行，encoding='utf-8'防乱码）
with open('test_orders.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['订单号', '商品', '金额', '城市'])
    writer.writeheader()     # 写表头行
    writer.writerows(orders) # 写数据行

# 读CSV（DictReader自动把每行转dict——⚠️数字是字符串！）
with open('test_orders.csv', 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        print(f"原始: {row}")
        print(f"金额类型={type(row['金额'])}, 值={row['金额']}")  # str!
        amount = int(row['金额'])  # ⚠️ 必须手动转int
        print(f"转换后: {amount}, 类型={type(amount)}")
        break


原始: {'订单号': 'ORD001', '商品': '蓝牙耳机', '金额': '299', '城市': '北京'}
金额类型=<class 'str'>, 值=299
转换后: 299, 类型=<class 'int'>


In [16]:
# ========== 【CSV小练】==========
# 任务：
# 1. 把下面3条订单用csv.DictWriter写入 test2.csv
# 2. 用csv.DictReader读回来
# 3. 打印第一行，检查金额的类型→在注释里写"类型是___"
# 4. 修正：把读回来的金额转成int，计算总金额

orders_new = [
    {"订单号": "D01", "商品": "键盘", "金额": 320, "城市": "深圳"},
    {"订单号": "D02", "商品": "鼠标", "金额": 150, "城市": "广州"},
    {"订单号": "D03", "商品": "耳机", "金额": 89,  "城市": "深圳"},
]

# 👇 你的代码：
import csv

with open("test2.csv","w",encoding= "utf-8") as f:
    writer = csv.DictWriter(f,fieldnames=["订单号","商品","金额","城市"])
    writer.writeheader()
    for order in orders_new:
        writer.writerow(order)

with open("test2.csv","r",encoding= 'utf-8') as f:
    for row in csv.DictReader(f):
        print(type(row["金额"]))
        print(f"总金额：{sum([int(x["金额"]) for x in orders_new])}")
        break


<class 'str'>
总金额：559


---

## 概念三：JSON处理——程序间对话的"世界语"

### 🤔 CSV很好，但嵌套结构怎么办？

CSV是扁平的——一行一条记录，每列一个值。但小王给你的产品配置文件长这样：

```json
{
  "产品ID": "BT-001",
  "折扣": {"满减": {"满": 200, "减": 30}, "打折": 0.85},
  "适用地区": ["北京", "上海", "深圳"]
}
```

这有嵌套——折扣里套了满减和打折，适用地区是一个列表。CSV处理不了这种"一个单元格里装一个dict"的结构。`json`模块就是专门处理这种嵌套数据的。


### JSON处理

**① 说白了就是**：JSON就是把Python的dict/list写成文本，任何程序（网页/Java/Go/Python）都能读。`json.load(f)` = JSON文本 → Python对象（dict/list）。`json.dump(obj, f)` = Python对象 → JSON文本。`json.loads(str)` = JSON字符串 → Python对象（s = string）。`json.dumps(obj)` = Python对象 → JSON字符串。四个函数，两两对称，load是"吸进来"，dump是"吐出去"。

**② 在沙盘里怎么用**：小王的`config.json`配置了产品折扣规则 → `config = json.load(f)` → 得到一个嵌套dict → `config['折扣']['满减']['满']`拿到满减值。分析结果要传给前端网页 → `json.dump(result, f, ensure_ascii=False, indent=2)` → 中文不转义，缩进2格人类可读。

**③ 跟你D1学的dict/list的关系**：JSON和Python类型的映射非常自然——JSON的`{}` = Python dict，JSON的`[]` = Python list，JSON的`"string"` = Python str，JSON的`123` = Python int/float，JSON的`true/false/null` = Python `True/False/None`。你D1学容器的时候实际上就在学JSON的数据结构！

**④ 为什么这样设计**：
- **`ensure_ascii=False`很重要**：默认`ensure_ascii=True`会把中文转成`\uXXXX`格式（如`\u5317\u4eac`）。人类完全看不懂。加了`False`，中文原样输出。
- **JSON的key必须用双引号**：`{"name": "张三"}`是合法JSON，`{'name': '张三'}`不是。这是JSON规范，不是Python的锅。
- **tuple在JSON里不存在**：JSON没有tuple类型。你在Python里`json.dump((1,2,3))` → JSON里变成`[1,2,3]`（list）。读回来时也是list，不会自动恢复成tuple。注意这个不对称！
- **`indent=2`让JSON人类可读**：不加indent，整个JSON挤在一行——5000条记录没法看。加了indent，每个嵌套层级缩进2格。


In [ ]:
# === JSON处理——它长这样 ===
import json

# 模拟小王的配置文件
config = {
    "产品ID": "BT-001",
    "产品名": "蓝牙耳机",
    "折扣": {"满减": {"满": 200, "减": 30}, "打折": 0.85},
    "适用地区": ["北京", "上海", "深圳"]
}

# 写JSON（ensure_ascii=False让中文可读，indent=2格式化）
with open('config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

# 读JSON
with open('config.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)
print(f"满减规则: 满{loaded['折扣']['满减']['满']}减{loaded['折扣']['满减']['减']}")

# loads/dumps：字符串↔对象（API通信常用）
json_str = '{"status": "ok", "count": 42}'
data = json.loads(json_str)       # 字符串→dict
print(data['count'])              # 42
print(json.dumps(data, ensure_ascii=False))  # dict→字符串


---

## 概念四：字符串方法全量——文本清洗的瑞士军刀

### 🤔 "299元"、" 北京 "、"待定"——你怎么处理这些脏数据？

小王给的CSV里，金额那一列是"299元"不是299，城市前后有空格，有的金额干脆写了"待定"。

如果你不用字符串方法，你可能写出这样的代码：
```python
# 手动去掉"元"字：
if '元' in amount_str:
    amount_str = amount_str[:-1]  # 去最后一个字"元"
# 但如果金额是"299元"（3个字）vs"1299元"（5个字）vs"待定"（根本不是数字）？
```

手动处理这些边界情况 → 几十行脆弱代码。Python的字符串方法让你用一行代码搞定：`amount_str.strip().replace('元', '')`。


### 字符串方法全量

**① 说白了就是**：字符串方法给你一把"文本手术刀"——去空格（`.strip()`像削土豆皮）、切分合并（`.split()`像切面条、`.join()`像把面条接回去）、查找替换（`.find()`像Ctrl+F、`.replace()`像Ctrl+H）、判断类型（`.isdigit()`问"你全是数字吗？"）、大小写转换（`.lower()`小声说、`.upper()`大声喊）。f-string是格式化神器——`f"{金额:.2f}"`="保留两位小数"。

**② 在沙盘里怎么用**：
- `" 北京 ".strip()` → `"北京"`（去城市字段前后空格）
- `"299元".replace('元', '')` → `"299"`（去金额里的"元"字）
- `"ORD001,蓝牙耳机,299,北京".split(',')` → `['ORD001', '蓝牙耳机', '299', '北京']`
- `f"{order['订单号']}: {order['商品']} ¥{order['金额']:.2f}"` → 格式化报告行

**③ 跟你D3学的函数的关系**：字符串方法是清洗流水线的基本操作。你D3学的函数封装，今天体现在：`def clean_amount(raw): return int(raw.strip().replace('元', ''))` —— 一行字符串方法链式调用 + 类型转换，封装成一个清洗函数。

**④ 为什么这样设计**：
- **字符串不可变——所有方法返回新字符串**：`s.strip()`不修改`s`，而是返回一个新的去除空格后的字符串。好处是你不小心传了一个字符串给函数，函数不可能偷偷改你的原始数据。
- **链式调用**：因为每个方法返回新字符串，你可以一口气写：`raw.strip().replace('元', '').replace(',', '')` —— 三步清洗一行搞定。
- **f-string比其他格式化方式更快更可读**：Python有三种字符串格式化——`%`格式化（C风格，老）、`.format()`（中间产物）、f-string（Python 3.6+，现代）。f-string直接内嵌表达式，运行速度比前两种都快。
- **`.split()` vs `.rsplit()` vs `.splitlines()`**：`.split()`从左开始分，`.rsplit()`从右开始分（可以限制分割次数），`.splitlines()`专门处理不同操作系统的换行符。


In [ ]:
# === 字符串方法——它们长这样 ===

# --- 去空格：strip/lstrip/rstrip ---
city_raw = "  深圳  "
print(f"|{city_raw.strip()}|")    # |深圳|

# --- 分割合并：split/join ---
csv_line = "ORD001,蓝牙耳机,299,北京"
parts = csv_line.split(',')
print(parts)                      # ['ORD001', '蓝牙耳机', '299', '北京']
print(' -> '.join(parts))         # ORD001 -> 蓝牙耳机 -> 299 -> 北京

# --- 查找替换：find/replace/count ---
text = "订单ORD001金额299元，订单ORD002金额1599元"
print(text.find('金额'))          # 8（第一次出现的位置，找不到返回-1）
print(text.replace('元', ''))     # 去掉所有"元"字
print(text.count('订单'))         # 2

# --- 判断类：startswith/endswith/isdigit/isalpha ---
print("ORD001".startswith("ORD"))   # True
print("299".isdigit())              # True（全是数字）
print("299元".isdigit())            # False（有"元"字）

# --- 大小写：lower/upper/title ---
print("hello world".title())        # Hello World

# --- f-string格式化 ---
amount, ratio, name = 299, 0.787, "蓝牙耳机"
print(f"¥{amount:.2f}")             # ¥299.00（2位小数）
print(f"{name:>12}")               # '      蓝牙耳机'（右对齐12位）
print(f"占比: {ratio:.1%}")         # 占比: 78.7%


In [ ]:
# ========== 【字符串小练】==========
# 任务：清洗下面的脏数据，输出干净的list of dict
# 要求：
#   - 城市字段去前后空格
#   - 商品字段如果为空（''），填'未知商品'
#   - 用f-string打印："{订单号}: {商品} (清洗后)"

dirty_data = [
    {"订单号": "ORD001", "商品": "蓝牙耳机", "城市": " 北京 "},
    {"订单号": "ORD002", "商品": "",          "城市": "上海"},
    {"订单号": "ORD003", "商品": "手机壳",   "城市": "  深圳  "},
]

# 👇 你的代码：



---

## 概念五：正则表达式——文本里的"X光机"

### 🤔 字符串方法能处理"找所有手机号"吗？

小王让你从用户行为日志里提取所有手机号和邮箱。日志长这样：

```
"用户张三(13812345678)于08:30登录，邮箱zhang@abc.com；
 用户李四(13987654321)于09:15下单，联系邮箱lisi@def.net"
```

用字符串方法怎么办？你得写：找到每个`(`的位置 → 往后取11位 → 判断是不是1开头全是数字 → ...。更麻烦的是——手机号有不同的号段（13x/15x/18x/19x），邮箱有各种域名。字符串方法处理"模式匹配"极其笨重。

**正则表达式就是专门做"模式匹配"的——你告诉它"手机号的模式是什么"，它帮你从任何文本里揪出来。**


### 正则表达式

**① 说白了就是**：正则是一种"搜索模式语言"——你描述"我要找的东西长什么样"，正则引擎帮你在文本里找到它。就像你跟警犬说"闻这个味道"（给模式），它帮你满仓库找（匹配文本）。不用正则 → 几十行if+字符串方法手动解析。用正则 → 一行`re.findall(r'1[3-9]\d{9}', text)`提取所有手机号。

**② 在沙盘里怎么用**：
- 从日志提取手机号：`re.findall(r'1[3-9]\d{9}', log_text)` → 一行拿到所有手机号
- 清洗金额字段：`re.sub(r'[^0-9.]', '', '299元')` → 得到`'299'`（删除一切不是数字和小数点的字符）
- 验证订单号格式：`re.search(r'^ORD\d{3}$', order_id)` → 检查是否是"ORD+3位数字"格式

**③ 跟你D2学的for/if的关系**：很多for+if做的字符串解析，正则一行就能替代。比如"找出所有以'Error:'开头的行"——for遍历+if startswith能做，但`re.findall(r'^Error:.*', text, re.MULTILINE)`一行搞定。正则不会取代for/if，但它是文本处理领域的最佳工具。

**④ 为什么这样设计**：
- **正则是一种DSL（领域特定语言）**：专门为"模式匹配"设计的迷你语言——和Python是两套语法。所以写在字符串里传给re模块（如`r'\d+'`），而不是Python语法的一部分。`r''`原始字符串很重要——不用`r''`的话，`\d`会被Python先解释成转义字符，再传给正则引擎，等于双重转义。
- **核心元字符速查**：`.`任意字符、`^`开头、`$`结尾、`*`0-n次、`+`1-n次、`?`0或1次、`{n,m}`n到m次、`[...]`字符集、`[^...]`排除集、`|`或、`()`捕获组、`\d`数字、`\w`字母数字下划线、`\s`空白、`\b`单词边界。
- **贪婪vs非贪婪——今天最重要的概念之一**：`.*`（贪婪）= 尽可能多吃字符。`.*?`（非贪婪）= 尽可能少吃。对`'<a><b>'`，`<.*>`匹配整个字符串（贪：从第一个`<`吃到最后一个`>`），`<.*?>`匹配`<a>`和`<b>`（不贪：遇到第一个`>`就停）。这个区别在爬虫/HTML解析中极其重要。
- **分组`()`两大用途**：①捕获——`re.search(r'(\d+)元', '299元').group(1)` → `'299'`（括号里的内容被单独取出）。②反向引用——`r'(\w+)\s+\1'`匹配重复单词如"the the"。`(?:...)`是"不捕获的分组"——只用于优先级但不保存，省内存。
- **前瞻后顾——匹配但不消费**：`(?=...)`后面必须跟...、`(?<=...)`前面必须是...。比如`r'\d+(?=元)'`匹配"299元"中的299，但"元"字不参与匹配。这在"我只想要金额数字但不要'元'字"的场景极其实用。
- **预编译`re.compile()`**：如果同一个正则要反复用（比如在循环里），先用`pattern = re.compile(r'...')`编译一次，然后用`pattern.search(text)`。避免每次调用都重新编译——在大量文本处理时性能差几倍。
- **⚠️`re.match` vs `re.search`**：`re.match`只从字符串开头匹配——如果开头不匹配，后面有也返回None。`re.search`在任意位置匹配。**除非你确定模式一定在开头，否则永远用`re.search`或`re.findall`。**


In [ ]:
# === 正则表达式——它长这样 ===
import re

text = '联系：张三13812345678，李四13987654321；邮箱zhang@abc.com, lisi@def.net'
html = '<a><b>内容1</b><b>内容2</b></a>'

# --- search：找第一个匹配 ---
m = re.search(r'1[3-9]\d{9}', text)  # 手机号模式
print(m.group() if m else '未找到')    # 13812345678

# --- findall：找所有匹配 ---
phones = re.findall(r'1[3-9]\d{9}', text)
print(phones)  # ['13812345678', '13987654321']

emails = re.findall(r'[\w.]+@[\w.]+', text)
print(emails)  # ['zhang@abc.com', 'lisi@def.net']

# --- sub：替换 ---
print(re.sub(r'元', '', '299元'))  # 299

# --- 贪婪 vs 非贪婪 ---
print(re.findall(r'<.*>', html))   # ['<a><b>内容1</b><b>内容2</b></a>'] 贪：全吃
print(re.findall(r'<.*?>', html))  # ['<a>', '<b>', '</b>', '<b>', '</b>', '</a>'] 不贪

# --- 分组捕获 ---
m = re.search(r'(\d+)元', '金额299元')
print(m.group(1))  # 299（group(0)是整个匹配'299元'，group(1)是括号里的'299'）

# --- 前瞻：只匹配"后面跟着'(不含税)'的金额" ---
prices = '价格:299元(含税),价格:1599元(不含税)'
print(re.findall(r'\d+(?=元\(不含税\))', prices))  # ['1599']

# --- compile预编译（频繁使用场景） ---
phone_pattern = re.compile(r'1[3-9]\d{9}')
print(phone_pattern.findall(text))  # ['13812345678', '13987654321']


In [ ]:
# ========== 【正则小练】==========
# 任务：从下面行程备注中提取信息

notes = [
    "实付299元,优惠50元,联系司机13812345678",
    "行程费150.5元(含高速费),可开发票",
    "实付金额: 待定, 司机电话 13987654321",
    "已付88.88元，司机邮箱 abc@diditaxi.com",
]

# 1. 对每条备注，用re.findall提取金额数字（"XX元"模式）
#    提示：r'(\d+\.?\d*)元'
# 2. 对每条备注，用re.findall提取手机号（1开头的11位数字）
# 3. 对每条备注，用re.findall提取邮箱地址（如果存在）

# 👇 你的代码：



In [ ]:
# ========== 基础练习1：文件读写——写出去、读回来、计算 ==========
# 场景：速卖通(电商) | 用到的武器：with open + write + read（新）
# 覆盖教学规格知识点：文件读写基础、'w'/'r'模式、encoding='utf-8'
#
# 任务：
# 1. 用'w'模式打开 numbers.txt → 写入 "100\n200\n300\n400\n500"（5个数字每行一个）
# 2. 用'r'模式打开 numbers.txt → read()读全部 → split()分割 → 转int → 求总和和平均值
# 3. 打印："总和: X, 平均值: Y"
# 提醒：别忘了encoding='utf-8'！
#
# 预期输出：总和: 1500, 平均值: 300.0

# 👇 在下面写你的代码：



In [ ]:
# ========== 基础练习2：CSV读写 + 格式化输出 ==========
# 场景：速卖通(电商) | 用到的武器：csv.DictWriter/DictReader + f-string（新）
# 覆盖教学规格知识点：CSV处理、f-string格式化、CSV数字是字符串陷阱
#
orders = [
    {"订单号": "ORD001", "商品": "蓝牙耳机", "品类": "数码", "金额": 299, "城市": "北京"},
    {"订单号": "ORD002", "商品": "机械键盘", "品类": "数码", "金额": 1599, "城市": "上海"},
    {"订单号": "ORD003", "商品": "运动鞋",   "品类": "服饰", "金额": 159, "城市": "深圳"},
    {"订单号": "ORD004", "商品": "充电宝",   "品类": "数码", "金额": 89,  "城市": "广州"},
    {"订单号": "ORD005", "商品": "连衣裙",   "品类": "服饰", "金额": 520, "城市": "北京"},
]
#
# 任务1：用csv.DictWriter把orders写入 orders_out.csv
#   （表头：订单号,商品,品类,金额,城市）
# 任务2：用csv.DictReader读回来
#   → 注意检查金额的类型→转int
#   → 用f-string格式化打印每行：
#     "ORD001 | 蓝牙耳机 | ¥  299.00 | 北京"
#   （订单号10位左对齐，商品10位左对齐，金额10位右对齐2位小数，城市）
# 任务3：计算总金额并打印
#
# 预期输出最后一行：总金额: 2666

# 👇 在下面写你的代码：



In [ ]:
# ========== 进阶练习1：游戏日志JSON解析 ==========
# 场景：**手游工作室（游戏行业）** | 用到的武器：json（新）+ dict/for/if（旧）
# 覆盖教学规格知识点：JSON处理、嵌套dict解析、json.load
#
# 下面模拟一份游戏事件日志JSON的内容：
import json

event_log = [
    {"event": "login",    "player": "暗影剑客", "time": "08:30", "ip": "192.168.1.1"},
    {"event": "battle",   "player": "暗影剑客", "result": "win",  "score": 150},
    {"event": "purchase", "player": "冰霜法师", "item": "宝石",   "amount": 68},
    {"event": "login",    "player": "冰霜法师", "time": "09:15", "ip": "192.168.1.2"},
    {"event": "battle",   "player": "风暴弓手", "result": "lose", "score": 30},
    {"event": "purchase", "player": "暗影剑客", "item": "皮肤",   "amount": 128},
    {"event": "login",    "player": "暗影剑客", "time": "12:00", "ip": "192.168.1.1"},
    {"event": "battle",   "player": "冰霜法师", "result": "win",  "score": 200},
]
# 实际场景：with open('event_log.json','r') as f: event_log = json.load(f)
#
# 任务1：用dict统计每种event类型的次数
# 任务2：统计每个玩家的purchase总金额（只算event=='purchase'的）
# 任务3：找出login次数最多的玩家，打印："登录王：XX(N次)"
# 任务4（附加）：统计每个玩家的battle胜率（win次数/总battle次数）
#
# 预期输出：
# 事件统计：{'login': 3, 'battle': 3, 'purchase': 2}
# 充值：暗影剑客196元，冰霜法师68元
# 登录王：暗影剑客(2次)

# 👇 在下面写你的代码：



In [ ]:
# ========== 进阶练习2：正则提取 + 字符串清洗组合 ==========
# 场景：**滴滴出行（出行行业）** | 用到的武器：re.findall/re.sub + str方法（新）
# 覆盖教学规格知识点：正则表达式、贪婪vs非贪婪、分组捕获、字符串方法
#
# 行程备注字段里什么乱七八糟的都有：
trip_notes = [
    "【行程】实付299元,优惠50元,联系司机13812345678,好评",
    "行程费150.5元(含高速费),可开发票,司机工号D12345",
    "实付金额: 待定, 司机电话 13987654321, 请尽快处理, 订单号TX20240115",
    "已付88.88元，找零麻烦，司机微信 abc@diditaxi.com, 行程编号:TX99001",
    "乘客取消, 退款 200 元, 原订单 TX88002, 投诉电话400-8820-1234",
]
#
# 任务1：对每条备注，提取金额数字（"XX元"模式，注意有的中间有空格）
#   提示：r'(\d+\.?\d*)\s*元'
# 任务2：对每条备注，提取手机号（1开头的11位数字）
# 任务3：对每条备注，提取订单号（"TX"开头后跟数字的模式）
#   提示：r'TX\d+'
# 任务4：对每条备注，提取400客服电话（400-XXXX-XXXX模式）
#   提示：r'400-\d{4}-\d{4}'
# 任务5：计算所有"已成功提取到金额"的订单的总金额
#
# 预期输出示例（第1条）：金额['299','50']，手机['13812345678']，订单号[]，400电话[]

# 👇 在下面写你的代码：



In [ ]:
# ========== 挑战练习1：读脏CSV→清洗→写干净CSV（完整流水线）==========
# 场景：速卖通(电商) | 用到的武器：csv + 字符串方法 + try/except + 正则——全套武器
# 覆盖教学规格知识点：文件IO+CSV+字符串+正则+try/except 综合
#
# 小王发给你的CSV有各种脏数据问题。下面模拟这个脏文件：
import csv, re

dirty_csv_lines = [
    "订单号,商品,金额,城市",
    "ORD001,蓝牙耳机,299元, 北京 ",
    "ORD002,,待定,上海",
    "ORD003,手机壳,89,北京",
    "",                                    # 空行！
    "ORD004,充电宝,499,  深圳  ",
    "ORD005,显示器,2599元,上海",
    "ORD006,数据线,  39.9 ,广州",
    "ORD007,鼠标,已退款,杭州",              # 金额不可用！
]
#
# 脏在哪里：
#   ① 金额里有"元"字或前后空格
#   ② 有的金额是"待定"/"已退款"等不可用的文字
#   ③ 城市前后有空格
#   ④ 有空白行
#   ⑤ 有的商品是空的
#
# 清洗规则：
#   金额去"元"字+去空格→转float（"待定"/"已退款"→None）
#   城市去首尾空格
#   跳过空行
#   商品为空→填"未知商品"
#   如果某行缺列（字段数<4）→跳过并打印警告
#
# 任务：
# 1. 把dirty_csv_lines写成实际的脏CSV文件 dirty_orders.csv
# 2. 读回→逐行清洗→收集干净的list of dict
# 3. 用csv.DictWriter把干净数据写入 clean_orders.csv
# 4. 打印清洗统计：原始行数/跳过行数/干净行数/金额为None的行
#
# ⚠️ 至少试15分钟。卡住了在注释里写下你的清洗思路。

# 👇 在下面写你的代码：



In [ ]:
# ========== 挑战练习2：正则深层应用——贪婪/分组/前瞻/后顾 ==========
# 场景：通用 | 用到的武器：re（贪婪vs非贪婪/分组捕获/前瞻后顾/反向引用）
# 覆盖教学规格知识点：正则贪婪vs非贪婪、()分组、(?:)不捕获、\1反向引用、(?=)/(?<=)前瞻后顾
#
import re

# === 任务A：贪婪 vs 非贪婪 ===
html = '<div><p>段落1</p><p>段落2</p></div>'
# 分别用贪婪和非贪婪匹配所有<p>...</p>标签
# 在注释里解释：为什么两个结果不同？

# 👇 你的代码：


# === 任务B：分组捕获 ===
order_str = '订单号:ORD001,金额:299元,城市:北京,日期:2024-01-15'
# 用分组一次性提取订单号、金额、城市三个值
# 打印："北京消费299元"

# 👇 你的代码：


# === 任务C：前瞻——只匹配但不消费 ===
price_str = '原价:299元(已售罄),现价:1599元(热卖),会员价:899元(限时)'
# 只提取"后面跟着'(热卖)'的金额"→预期：1599
# 提示：r'\d+(?=元\(热卖\))'

# 👇 你的代码：


# === 任务D：后顾——"前面是XX的YY" ===
log = '[ERROR]磁盘满 [INFO]任务完成 [ERROR]连接超时 [WARN]内存高 [ERROR]权限拒绝'
# 只提取[ERROR]后面的错误信息（不含[ERROR]本身）
# 预期：['磁盘满', '连接超时', '权限拒绝']
# 提示：r'(?<=\[ERROR\])[^\[]+'

# 👇 你的代码：


# === 任务E（选做）：反向引用——找重复单词 ===
text = 'the the problem is that we we need to fix this this bug'
# 用反向引用找出所有连续重复的单词
# 提示：r'(\w+)\s+\1'

# 👇 你的代码：



In [ ]:
# ========== 迁移练习：学籍系统——读成绩JSON→计算→写回新JSON ==========
# 场景：**在线教育（全新领域）** | 你需要自己判断今天学的哪些武器
# 覆盖教学规格知识点：JSON读写、字符串处理、f-string格式化、函数封装——全综合
#
# 给一个学生成绩JSON（这里用Python对象模拟，实际场景从文件读）：
import json

students_data = [
    {"学号": "S01", "姓名": "张三", "语文": 85, "数学": 92, "英语": 78},
    {"学号": "S02", "姓名": "李四", "语文": 78, "数学": 88, "英语": 85},
    {"学号": "S03", "姓名": "王五", "语文": 95, "数学": 76, "英语": 82},
    {"学号": "S04", "姓名": "赵六", "语文": 68, "数学": 85, "英语": 90},
    {"学号": "S05", "姓名": "陈七", "语文": 88, "数学": 91, "英语": 76},
]
#
# 任务1：把students_data写入实际的JSON文件 students.json
#   → 再读回来 → 验证读回来的和原始数据一致
#
# 任务2：计算每个学生的：
#   - 总分 = 语文+数学+英语
#   - 均分 = 总分/3（保留1位小数）
#   - 等级 = 均分>=90→'A' / >=80→'B' / >=70→'C' / >=60→'D' / <60→'F'
#   - 排名（均分相同→并列排名）
#   把这些字段加到每个学生的dict里
#
# 任务3：把带排名和等级的结果写入 students_with_rank.json
#
# 任务4：用f-string格式化打印前3名学生：
#   "第N名：姓名 | 总分：XX | 均分：XX.X | 等级：X"
#
# 任务5（附加）：计算班级统计——
#   全班语文/数学/英语平均分、A等级人数和占比
#   写到 ranking_summary.txt 文件里
#
# 你需要自己判断今天学的哪个武器能解决每个子问题。

# 👇 在下面写你的代码：



---

## ⚠️ 新手最容易踩的坑

### 坑1：`'w'`模式一打开就清空原文件 + 忘写`encoding='utf-8'`

❌ 错误写法：
```python
# 只想看看文件内容，结果把文件清空了！
with open('important_data.csv', 'w') as f:  # 'w'一开就清空！
    pass  # 什么都没写，但文件已经空了！

# Windows中文乱码
with open('data.csv', 'r') as f:  # 默认GBK，UTF-8文件→乱码
    content = f.read()
```

✅ 正确写法：
```python
# 只想读→用'r'。想追加→用'a'。只有确实要覆盖才用'w'
with open('important_data.csv', 'r', encoding='utf-8') as f:
    content = f.read()  # 安全——不会清空

# 写CSV也要注意
with open('output.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=[...])
```

**为什么错**：`'w'`模式的设计意图是"覆盖写入"——打开瞬间清空文件。`encoding`不写→Python用系统默认编码（Windows=GBK）→UTF-8文件里的中文变成乱码。

---

### 坑2：CSV读进来的数字是字符串！

❌ 错误写法：
```python
for row in csv.DictReader(f):
    total += row['金额']  # 😱 字符串拼接！'299' + '89' = '29989'
```

✅ 正确写法：
```python
for row in csv.DictReader(f):
    total += int(row['金额'])  # 先转int再做数学运算
```

**为什么错**：CSV文件是纯文本格式，没有类型信息。`csv.DictReader`读进来的所有字段都是字符串，即使它看起来像数字。必须手动`int()`或`float()`转换。

---

### 坑3：`re.match`只从开头匹配——找不到就返回None

❌ 错误写法：
```python
text = 'abc123def'
result = re.match(r'\d+', text)  # None！因为开头是'abc'不是数字
```

✅ 正确写法：
```python
result = re.search(r'\d+', text)   # 找到123——在任意位置搜索
# 或者
results = re.findall(r'\d+', text) # ['123']——找所有匹配
```

**为什么错**：`re.match`的设计语义是"从字符串开头尝试匹配"——相当于隐式地在模式前面加了`^`。除非你确定模式一定在开头（如验证订单号格式`r'^ORD\d{3}$'`），否则用`re.search`。

---

### 坑4：贪婪匹配`.*`拿到比预期多的内容

❌ 错误写法：
```python
html = '<div>标题</div><div>内容</div>'
re.findall(r'<div>.*</div>', html)
# → ['<div>标题</div><div>内容</div>']
# 😱 两个div被当成一个了！.* 贪心地从第一个<div>吃到最后一个</div>
```

✅ 正确写法：
```python
re.findall(r'<div>.*?</div>', html)
# → ['<div>标题</div>', '<div>内容</div>']
# .*? 非贪婪：遇到第一个</div>就停下来
```

**为什么错**：`*`和`+`默认是贪婪的——尽可能多地匹配。在后面加`?`变成非贪婪——尽可能少地匹配。在HTML/XML解析、引号内内容提取等场景，非贪婪是正确答案。

> **口诀**：`.*`是贪吃蛇——从头吃到尾。`.*?`是挑食者——吃一口就停。


---

## 📝 参考答案

> **重要**：先自己写完所有练习再看答案。看着答案觉得自己"会了"是最大的幻觉。

### 文件读写小练
```python
with open('numbers.txt', 'w', encoding='utf-8') as f:
    f.write('100\n200\n300')
with open('numbers.txt', 'r', encoding='utf-8') as f:
    nums = [int(x) for x in f.read().split()]
    print(f'总和：{sum(nums)}')  # 总和：600
```

### CSV小练
```python
with open('test2.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['订单号','商品','金额','城市'])
    w.writeheader()
    w.writerows(orders_new)
total = 0
with open('test2.csv', 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        print(row)  # 金额是str！
        total += int(row['金额'])
        break
print(f'总金额: {total}')  # 559
```

### 字符串小练
```python
for item in dirty_data:
    item['城市'] = item['城市'].strip()
    if not item['商品']:
        item['商品'] = '未知商品'
    print(f"{item['订单号']}: {item['商品']} (清洗后)")
```

### 正则小练
```python
for note in notes:
    amounts = re.findall(r'(\d+\.?\d*)元', note)
    phones = re.findall(r'1[3-9]\d{9}', note)
    emails = re.findall(r'[\w.]+@[\w.]+', note)
    print(f"{note[:30]}... → 金额{amounts}, 手机{phones}, 邮箱{emails}")
```

### 基础练习1
```python
with open('numbers.txt', 'w', encoding='utf-8') as f:
    f.write('100\n200\n300\n400\n500')
with open('numbers.txt', 'r', encoding='utf-8') as f:
    nums = [int(x) for x in f.read().split()]
print(f'总和: {sum(nums)}, 平均值: {sum(nums)/len(nums)}')
```

### 基础练习2
```python
with open('orders_out.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['订单号','商品','品类','金额','城市'])
    w.writeheader()
    w.writerows(orders)
total = 0
with open('orders_out.csv', 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        amt = int(row['金额'])
        total += amt
        print(f"{row['订单号']:<10} | {row['商品']:<10} | ¥{amt:>10.2f} | {row['城市']}")
print(f'总金额: {total}')
```

### 进阶练习1
```python
from collections import Counter
event_count = Counter(e['event'] for e in event_log)
purchase_by_player = {}
for e in event_log:
    if e['event'] == 'purchase':
        purchase_by_player[e['player']] = purchase_by_player.get(e['player'], 0) + e['amount']
login_count = Counter(e['player'] for e in event_log if e['event'] == 'login')
top_login = login_count.most_common(1)[0]
print(f"事件统计：{dict(event_count)}")
for p, amt in purchase_by_player.items():
    print(f"{p}：{amt}元")
print(f"登录王：{top_login[0]}({top_login[1]}次)")
# 附加：胜率
battles_by_player = {}
for e in event_log:
    if e['event'] == 'battle':
        p = e['player']
        if p not in battles_by_player:
            battles_by_player[p] = {'win': 0, 'total': 0}
        battles_by_player[p]['total'] += 1
        if e['result'] == 'win':
            battles_by_player[p]['win'] += 1
for p, stats in battles_by_player.items():
    rate = stats['win'] / stats['total'] * 100
    print(f"{p}胜率: {rate:.1f}%")
```

### 进阶练习2
```python
for note in trip_notes:
    amounts = re.findall(r'(\d+\.?\d*)\s*元', note)
    phones = re.findall(r'1[3-9]\d{9}', note)
    orders = re.findall(r'TX\d+', note)
    hotline = re.findall(r'400-\d{4}-\d{4}', note)
    print(f"金额{amounts}, 手机{phones}, 订单{orders}, 热线{hotline}")
# 总金额（只算有金额的）
total = 0
for note in trip_notes:
    amts = re.findall(r'(\d+\.?\d*)\s*元', note)
    if amts:
        total += sum(float(a) for a in amts)
print(f"已提取总金额: {total}")
```

### 挑战练习1
```python
# 1. 写脏文件
with open('dirty_orders.csv', 'w', newline='', encoding='utf-8') as f:
    for line in dirty_csv_lines:
        f.write(line + '\n')

# 2. 清洗函数
def clean_amount(raw):
    raw = raw.strip().replace('元', '').replace(',', '')
    try:
        return float(raw)
    except ValueError:
        return None

clean_rows, skipped, none_amount = [], 0, 0
with open('dirty_orders.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if not any(row.values()):
            skipped += 1
            continue
        row['城市'] = row['城市'].strip()
        row['商品'] = row['商品'].strip() or '未知商品'
        amt = clean_amount(row['金额'])
        if amt is None:
            none_amount += 1
        row['金额'] = amt
        clean_rows.append(row)

# 3. 写干净CSV
with open('clean_orders.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['订单号','商品','金额','城市'])
    w.writeheader()
    w.writerows(clean_rows)

print(f"原始{len(dirty_csv_lines)-1}行→跳过{skipped}行→干净{len(clean_rows)}行→金额异常{none_amount}行")
```

### 挑战练习2
```python
# A: 贪婪vs非贪婪
print(re.findall(r'<p>.*</p>', html))    # ['<p>段落1</p><p>段落2</p>'] 贪
print(re.findall(r'<p>.*?</p>', html))   # ['<p>段落1</p>', '<p>段落2</p>'] 不贪
# .* 从第一个<p>吃到最后一个</p>；.*? 遇到第一个</p>就停

# B: 分组
m = re.search(r'订单号:(\w+),金额:(\d+)元,城市:(\w+)', order_str)
if m:
    print(f"{m.group(3)}消费{m.group(2)}元")  # 北京消费299元

# C: 前瞻
print(re.findall(r'\d+(?=元\(热卖\))', price_str))  # ['1599']

# D: 后顾
print(re.findall(r'(?<=\[ERROR\])[^\[]+', log))
# ['磁盘满 ', '连接超时 ', '权限拒绝']

# E: 反向引用
print(re.findall(r'(\w+)\s+\1', text))  # ['the', 'we', 'this']
```

### 迁移练习
```python
# 1. 写+读JSON
with open('students.json', 'w', encoding='utf-8') as f:
    json.dump(students_data, f, ensure_ascii=False, indent=2)
with open('students.json', 'r', encoding='utf-8') as f:
    students = json.load(f)
print("验证:", students == students_data)  # 应该True

# 2. 计算
for s in students:
    s['总分'] = s['语文'] + s['数学'] + s['英语']
    s['均分'] = round(s['总分'] / 3, 1)
    avg = s['均分']
    if avg >= 90:
        s['等级'] = 'A'
    elif avg >= 80:
        s['等级'] = 'B'
    elif avg >= 70:
        s['等级'] = 'C'
    elif avg >= 60:
        s['等级'] = 'D'
    else:
        s['等级'] = 'F'
students.sort(key=lambda s: -s['均分'])
for i, s in enumerate(students, 1):
    s['排名'] = i
    if i <= 3:
        print(f"第{i}名：{s['姓名']} | 总分：{s['总分']} | 均分：{s['均分']} | 等级：{s['等级']}")

# 3. 写回
with open('students_with_rank.json', 'w', encoding='utf-8') as f:
    json.dump(students, f, ensure_ascii=False, indent=2)

# 5. 班级统计
subjects = ['语文', '数学', '英语']
summary_lines = []
for subj in subjects:
    avg = sum(s[subj] for s in students) / len(students)
    summary_lines.append(f"{subj}平均分: {avg:.1f}")
a_count = sum(1 for s in students if s['等级'] == 'A')
summary_lines.append(f"A等级: {a_count}人({a_count/len(students)*100:.1f}%)")
with open('ranking_summary.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_lines))
print("班级统计已写入 ranking_summary.txt")
```

---

## 🩺 偏差诊断

| 如果你的输出/报错是... | 你可能的误解是... | 补救 |
|----------------------|-----------------|------|
| 读文件时中文乱码（如乱码字符） | 忘了`encoding='utf-8'` | 在所有`open()`里加上`encoding='utf-8'` |
| `FileNotFoundError: [Errno 2]` | 路径写错了，或文件不在当前目录 | 用`import os; print(os.getcwd())`确认当前目录 |
| CSV的金额相加变成字符串拼接（`'299'+'89'='29989'`） | CSV读的数字是字符串，忘了int()转换 | 回到坑2。在`csv.DictReader`循环里立即`int(row['金额'])` |
| `re.match`返回None但数据里明明有 | 用了match而不是search——match只从开头匹配 | 回到坑3。改用`re.search`或`re.findall` |
| 正则匹配到太多内容（如整个HTML被当做一个标签） | 用了贪婪`.*`而非非贪婪`.*?` | 回到坑4。在量词后加`?`变非贪婪 |
| `PermissionError` 或文件被占用 | `'w'`模式打开了已在别处打开的文件 | 确保`with`语句正常退出；检查文件是否在Excel中打开 |
| `json.decoder.JSONDecodeError` | JSON文件格式有问题（单引号、尾逗号、注释） | JSON严格要求双引号、最后一项不能有逗号、不支持注释 |
| `TypeError: Object of type X is not JSON serializable` | 试图`json.dump`一个不可序列化的对象 | 转为可序列化类型：datetime→str、set→list |


In [ ]:
# ========== 🔙 回到开头的问题 ==========
#
# 还记得早上的认知冲突吗？小王给你一个脏CSV，5000行——
# 金额里有"元"字、"待定"，城市有空格，还有空行。
#
# 现在用今天学的全套武器，完成一条完整的"读→清洗→写"流水线：

import csv, re

# 模拟脏数据文件内容
dirty_lines = [
    "订单号,商品,金额,城市\n",
    "ORD001,蓝牙耳机,299元, 北京 \n",
    "ORD002,机械键盘,待定,上海\n",
    "ORD003,,89,北京\n",
    "\n",  # 空行
    "ORD004,充电宝,499,  深圳  \n",
    "ORD005,显示器,2599元,上海\n",
]

# 任务：
# 1. 写入dirty.csv
# 2. 读→清洗（金额去"元"转int，"待定"→None，城市去空格，空商品→"未知"，跳空行）
# 3. 写入clean.csv
# 4. 打印："清洗完成：X条干净数据，Y条异常数据"
#
# 如果做不出→回看挑战练习1的参考答案，理解每一步在做什么

# 👇 在下面写你的完整清洗流水线：



---

## ✅ 今天你学会了吗？

- [ ] **初级（L2：能用）**：不看笔记，独立完成基础练习1-2
  没过 → 回看对应概念的 cell b-c，重做基础练习

- [ ] **中级（L3：能讲）**：下面写一段200字以内的白话，讲给不懂技术的同事听：

  > "with open()比手动f.close()好在哪里？CSV读进来的数字为什么是字符串？
  > 贪婪和非贪婪正则有什么区别？什么时候用re.compile？"

  *你的回答（在这里写）：*







  没过 → 你写不出来说明停留在L2，回到概念1/2/5的cell b看第④段

- [ ] **高级（L4：能判断）**：迁移练习独立完成了吗？挑战练习1的脏CSV你清洗干净了吗？
  没过 → 补做迁移练习，不许查笔记

---

### 📋 概念掌握检查表（对照今日知识地图逐项检验）

**文件读写基础**
- [ ] 能用 `with open(path, mode, encoding='utf-8')` 安全读写文件
- [ ] 能解释 `'r'` / `'w'` / `'a'` / `'r+'` 四种模式的区别
- [ ] 知道 `'w'` 模式一打开就清空文件内容
- [ ] 能用 `f.read()` / `f.readline()` / `f.readlines()` 读文件
- [ ] 能用 `f.write()` / `f.writelines()` 写文件
- [ ] 能解释为什么 `with` 比手动 `f.close()` 安全（异常安全）
- [ ] 知道写CSV加 `newline=''` 防止Windows多余空行

**CSV处理**
- [ ] 能用 `csv.DictReader(f)` 读CSV（每行自动变dict）
- [ ] 能用 `csv.DictWriter` + `writeheader()` + `writerows()` 写CSV
- [ ] **知道CSV读进来的数字是字符串，必须 `int()` / `float()` 转换**

**JSON处理**
- [ ] 能用 `json.load(f)` / `json.dump(obj, f)` 读写JSON文件
- [ ] 能用 `json.loads(str)` / `json.dumps(obj)` 处理JSON字符串
- [ ] 知道 `ensure_ascii=False` 让中文不转义
- [ ] 知道 `indent=2` 让JSON人类可读
- [ ] 知道 tuple 在JSON序列化时会变成 list（不对称！）

**字符串方法**
- [ ] 能用 `.strip()` / `.lstrip()` / `.rstrip()` 去空格
- [ ] 能用 `.split()` / `.join()` 分割合并
- [ ] 能用 `.find()` / `.replace()` / `.count()` 查找替换
- [ ] 能用 `.startswith()` / `.endswith()` / `.isdigit()` / `.isalpha()` 判断类型
- [ ] 能用 `.lower()` / `.upper()` / `.title()` 大小写转换
- [ ] 能用 f-string 格式化：`:.2f` / `:>10` / `:.1%`
- [ ] 能链式调用：`s.strip().lower().replace('元','')`

**正则表达式**
- [ ] 能用 `re.search()` 找第一个匹配 / `re.findall()` 找所有匹配
- [ ] 能用 `re.sub()` 替换 / `re.split()` 按正则分割
- [ ] 能写出常用元字符：`.` `\d` `\w` `\s` `^` `$` `*` `+` `?` `{n,m}` `[]` `()` `|`
- [ ] **能区分贪婪 `.*` vs 非贪婪 `.*?` 并解释为什么不同**
- [ ] 能用 `()` 分组捕获，知道 `group(1)` 取第一个括号
- [ ] 能用 `(?=...)` 正前瞻 / `(?<=...)` 正后顾（"匹配但不消费"）
- [ ] 能用 `re.compile()` 预编译（频繁使用场景）
- [ ] **知道 `re.match` 只从开头匹配 → 用 `re.search` 替代**
- [ ] 知道 `re.DOTALL` / `re.IGNORECASE` / `re.MULTILINE` 标志的作用
- [ ] 能解释为什么正则里用 `r''` 原始字符串（避免双重转义）


---

## 🔮 明天预告

**明天你将能解决**：前四天你写了大量代码——读文件、清洗文本、分析数据、格式化输出。现在小王说"把这些整理成一个正经项目，我要能一键运行，还要能传给其他同事用"。

**文件IO（今天） + 模块化（明天） = 你的代码从"一个长长的Notebook"变成"可复用的标准Python项目"。**

明天学：**代码组织 + 模块化 + 虚拟环境 + `.py` vs `.ipynb`**——数据分析师的"工程化"第一课。

---

### 🗺️ 今天的概念在本周知识网络中的位置

- **D3（昨天）→ D4（今天）**：用函数封装文件读写逻辑——`def read_orders(path):`、`def clean_data(rows):`、`def write_report(data, path):`。函数是文件IO的标准封装方式
- **D4（今天）→ D5（明天）**：今天的代码会把IO、清洗、分析都写在一个文件里 → 明天把它们拆成 `io.py` / `clean.py` / `analyze.py` 三个模块 → `main.py` 一键调用
- **D4（今天）→ D6（周六项目）**：周六你要读三份CSV、清洗、分析、输出报告 + 写README + Git提交——今天学的所有文件IO技能都会在项目中用到
- **D4（今天）→ W2（pandas）**：pandas的 `pd.read_csv()` 比csv模块强大得多——自动推断类型、处理缺失值、支持分块读取。但理解csv模块的底层原理，你才能理解pandas在帮你做什么
